In [1]:
import polars as pl
import pandas as pd
from openfe import OpenFE, transform, tree_to_formula   # ← 直接用官方 tree_to_formula
import os

In [2]:
# ──────────────────────────────────────────────────────────────────────────────
# 主函数：空间拆分 + OpenFE 特征生成 + 公式列名重命名
# ──────────────────────────────────────────────────────────────────────────────
def generate_openfe_features_with_spatial_split(
    train_df: pl.DataFrame,
    n_new_features: int = 100,
    split_ratio: float = 0.8,
    random_seed: int = 42,
) -> dict:
    """
    内部进行基于经纬度的空间拆分，防止 OpenFE 发生地理位置过拟合。
    传入全量 train_df，输出行数完全一致、但包含二次特征的三个 target 专用数据集。

    新特征列名为 OpenFE 生成的完整公式字符串，例如：
        GroupByThenMean(AHU, continent)
        (ppt * soil)
        residual(Dissolved Reactive Phosphorus)
    """

    num_cores = os.cpu_count() or 1
    print(f"🖥️  检测到系统 CPU 核心数为: {num_cores}，已分配给 OpenFE 并发计算。")

    # ── 1. 定义非特征列和预测目标 ──────────────────────────────────────────────
    meta_cols = ["Latitude", "Longitude", "Sample Date", "continent"]
    targets = [
        "Total Alkalinity",
        "Electrical Conductance",
        "Dissolved Reactive Phosphorus",
    ]

    # ── 2. 空间拆分（Spatial Split）─────────────────────────────────────────────
    unique_locs = train_df.select(["Latitude", "Longitude"]).unique()
    train_locs = unique_locs.sample(fraction=split_ratio, seed=random_seed)
    train_sub_df = train_df.join(train_locs, on=["Latitude", "Longitude"], how="inner")

    print(f"🌍 空间拆分完成：共有 {unique_locs.height} 个独特地点。")
    print(f"   - 用于 OpenFE 探索的地点数: {train_locs.height}（数据量: {train_sub_df.height} 行）")
    print(f"   - 全量数据集将作为最终输出（数据量: {train_df.height} 行）")

    # ── 3. 强类型清洗：Polars → pandas ──────────────────────────────────────────
    def robust_clean_and_cast(df: pl.DataFrame) -> pd.DataFrame:
        exprs = []
        for col_name in df.columns:
            if col_name in meta_cols or col_name in targets:
                exprs.append(pl.col(col_name))
                continue
            dtype = df.schema[col_name]
            if dtype in [pl.Utf8, pl.String]:
                expr = (
                    pl.when(
                        pl.col(col_name)
                        .str.to_lowercase()
                        .is_in(["null", "nan", "none", ""])
                    )
                    .then(None)
                    .otherwise(pl.col(col_name))
                    .cast(pl.Float64, strict=False)
                )
                exprs.append(expr.alias(col_name))
            elif dtype not in [pl.Float32, pl.Float64, pl.Int32, pl.Int64]:
                exprs.append(pl.col(col_name).cast(pl.Float64, strict=False).alias(col_name))
            else:
                exprs.append(pl.col(col_name).cast(pl.Float64).alias(col_name))
        return df.with_columns(exprs).to_pandas()

    train_sub_pd = robust_clean_and_cast(train_sub_df)
    full_pd = robust_clean_and_cast(train_df)

    results = {}
    cols_to_drop = meta_cols + targets

    # ── 4. 对每个目标分别跑 OpenFE ───────────────────────────────────────────────
    for target in targets:
        print(f"\n🚀 [OpenFE] 开始为回归目标: {target} 生成特征...")

        # A. 准备探索集（用于 fit）
        train_sub_x = train_sub_pd.drop(columns=cols_to_drop)
        train_sub_y = train_sub_pd[target]

        # B. 准备全量集（用于 transform）
        full_x = full_pd.drop(columns=cols_to_drop)
        full_y = full_pd[target]

        # C. 初始化并运行 OpenFE
        ofe = OpenFE()
        features = ofe.fit(
            data=train_sub_x,
            label=train_sub_y,
            task="regression",
            n_jobs=num_cores,
            n_data_blocks=1,
        )

        # D. 选出 Top-N 特征
        top_features = features[:n_new_features]

        # ★ 关键步骤 1：在 delete() 之前缓存所有公式字符串
        #   tree_to_formula() 需要完整的 Node 树结构，
        #   一旦调用 delete() 子节点引用会被清除，公式就无法再解析了
        formulas_cache = [tree_to_formula(f) for f in top_features]

        # ★ 关键步骤 2：参照 IEEE.py 调用 delete()
        #   delete() 把特征数据序列化到 feather 缓存文件，
        #   transform() 再从磁盘读取，避免全量数据在内存中重复计算
        for feat in top_features:
            feat.delete()

        # E. Transform 应用到全量数据集
        dummy_test = full_x.iloc[:1]
        full_x_trans, _ = transform(full_x, dummy_test, top_features, n_jobs=num_cores)

        # ── ★ 核心修复：用缓存的公式字符串构造唯一列名 ★ ─────────────────────────
        existing_set = set(full_x.columns)   # 原始列名集合，用于冲突检测
        formula_seen: dict[str, int] = {}
        rename_map: dict[str, str] = {}

        for i, formula in enumerate(formulas_cache):
            # 处理重复公式（同一公式多次出现时追加 _2, _3 ...）
            if formula in formula_seen:
                formula_seen[formula] += 1
                candidate = f"{formula}_{formula_seen[formula]}"
            else:
                formula_seen[formula] = 1
                candidate = formula

            # 处理与原有列名冲突（防御性处理）
            suffix = 2
            final_name = candidate
            while final_name in existing_set:
                final_name = f"{candidate}_{suffix}"
                suffix += 1

            existing_set.add(final_name)
            rename_map[f"autoFE_f_{i}"] = final_name

        full_x_trans = full_x_trans.rename(columns=rename_map)
        # ─────────────────────────────────────────────────────────────────────

        # 打印映射关系
        print(f"\n   📋 [{target}] 新特征列名映射（占位符 → 公式）：")
        for placeholder, formula_name in rename_map.items():
            print(f"      {placeholder:<15} → {formula_name}")

        # F. 重新拼回元数据和当前目标列
        for col in meta_cols:
            full_x_trans[col] = full_pd[col]
        full_x_trans[target] = full_y

        # G. 转回 Polars（此时列名已保证唯一，不会再报 ValueError）
        final_pl_df = pl.from_pandas(full_x_trans)

        results[target] = {
            "train": final_pl_df,
            "generated_features": top_features,
            "rename_map": rename_map,
            "formulas": formulas_cache,     # 原始公式列表，方便下游分析
        }

        print(f"\n✅ {target} 扩充完成！最终矩阵 Shape: {final_pl_df.shape}")

    return results

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 调用示例
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    train = pl.read_csv(r"D:\Haoxuan_Xu_Documents_Uni\EY_Challenge\data\train_engineered.csv")

    openfe_results = generate_openfe_features_with_spatial_split(train)


🖥️  检测到系统 CPU 核心数为: 32，已分配给 OpenFE 并发计算。
🌍 空间拆分完成：共有 464 个独特地点。
   - 用于 OpenFE 探索的地点数: 371（数据量: 12336 行）
   - 全量数据集将作为最终输出（数据量: 16025 行）

🚀 [OpenFE] 开始为回归目标: Total Alkalinity 生成特征...
The number of candidate features is 53998
Start stage I selection.


100%|██████████| 128/128 [01:24<00:00,  1.52it/s]


26415 same features have been deleted.
The number of remaining candidate features is 27464
Start stage II selection.


100%|██████████| 128/128 [00:24<00:00,  5.23it/s]


Finish data processing.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 2.062784 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6430054
[LightGBM] [Info] Number of data points in the train set: 9868, number of used features: 27580

   📋 [Total Alkalinity] 新特征列名映射（占位符 → 公式）：
      autoFE_f_0      → (dist_river_min_m-dist_wastewater_min_m)
      autoFE_f_1      → (def/evaporation_from_vegetation_transpiration)
      autoFE_f_2      → (lake_mix_layer_temperature-soil_temperature_level_3)
      autoFE_f_3      → freq(dist_river_min_m)
      autoFE_f_4      → GroupByThenRank(surface_pressure,river_count_1000m)
      autoFE_f_5      → (dist_mine_min_m-dist_wastewater_min_m)
      autoFE_f_6      → (tmax/leaf_area_index_high_vegetation)
      autoFE_f_7      → residual(dist_wastewater_min_m)
      autoFE_f_8      → (dist_river_min_m-dist_mine_min_m)
      autoFE_f_9      → (dist_river_min_m/leaf_area_index_

...

Finish data processing.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 2.080454 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6399927
[LightGBM] [Info] Number of data points in the train set: 9868, number of used features: 27513

   📋 [Electrical Conductance] 新特征列名映射（占位符 → 公式）：
      autoFE_f_0      → GroupByThenRank(volumetric_soil_water_layer_4,river_count_250m)
      autoFE_f_1      → freq(dist_river_min_m)
      autoFE_f_2      → max(lake_mix_layer_depth,log_dist_river_min_m)
      autoFE_f_3      → min(forecast_albedo,leaf_area_index_high_vegetation)
      autoFE_f_4      → max(log_dist_river_min_m,HCF)
      autoFE_f_5      → (leaf_area_index_high_vegetation-log_dist_river_min_m)
      autoFE_f_6      → (lake_mix_layer_depth/soil_phh2o_present)
      autoFE_f_7      → (forecast_albedo/leaf_area_index_low_vegetation)
      autoFE_f_8      → residual(dist_mine_min_m)
      autoFE_f_9      → (su

100%|██████████| 128/128 [01:12<00:00,  1.76it/s]


27424 same features have been deleted.
The number of remaining candidate features is 25257
Start stage II selection.


100%|██████████| 128/128 [00:29<00:00,  4.39it/s]


Finish data processing.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.932214 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5843140
[LightGBM] [Info] Number of data points in the train set: 9868, number of used features: 25372

   📋 [Dissolved Reactive Phosphorus] 新特征列名映射（占位符 → 公式）：
      autoFE_f_0      → (def/forest_pct_2000m)
      autoFE_f_1      → (lake_ice_temperature/forest_pct_2000m)
      autoFE_f_2      → (forecast_albedo-soil_clay_present)
      autoFE_f_3      → GroupByThenMax(surface_thermal_radiation_downwards_hourly,rain_days_30d)
      autoFE_f_4      → (volumetric_soil_water_layer_3/forest_pct_2000m)
      autoFE_f_5      → (surface_solar_radiation_downwards/surface_solar_radiation_downwards_hourly)
      autoFE_f_6      → (ws/forest_pct_2000m)
      autoFE_f_7      → (pdsi/ppt_lag1)
      autoFE_f_8      → (forecast_albedo/soil_temperature_level_3)
      autoFE_f_9      → GroupB

In [4]:
final_train_alkalinity = openfe_results["Total Alkalinity"]["train"]

In [7]:
final_train_alkalinity.write_csv(r'D:\Haoxuan_Xu_Documents_Uni\EY_Challenge\data\train_split\train_alkalinity.csv')

In [8]:
final_train_electrical_conductance = openfe_results["Electrical Conductance"]["train"]

In [10]:
final_train_electrical_conductance.write_csv(r'D:\Haoxuan_Xu_Documents_Uni\EY_Challenge\data\train_split\train_electrical_conductance.csv')

In [11]:
final_train_dissolved_reactive_phosphorus = openfe_results["Dissolved Reactive Phosphorus"]["train"]

In [12]:
final_train_dissolved_reactive_phosphorus.write_csv(r'D:\Haoxuan_Xu_Documents_Uni\EY_Challenge\data\train_split\train_dissolved_reactive_phosphorus.csv')